In [4]:
def my_agent(obs):
    market = []
    wheat_seeds = obs["private"]["seeds"].get("WHEAT", 0) # Revisa si hay semillas en el inventario primero
    if obs["step"] == 0 or wheat_seeds <= 0: # Si no hay semillas, las compra en el mercado
        market = [["BUY_SEED", "WHEAT", 1]]
    farmer_pos = obs["farms"][obs["player"]]["farmer"]
    x, y = farmer_pos
    tile = obs["farms"][obs["player"]]["tiles"][y][x]
    #print(tile)
    wheat_in_shed = obs["private"]["shed"].get("WHEAT", 0)
    if wheat_in_shed > 0:
        market = [["SELL","WHEAT",1]]

    if tile is None and wheat_seeds > 0: # Comprueba que no hay nada en la celda y que hay semilla en inventario
        farmer = ["PLANT", "WHEAT"] # Se planta la semilla
    elif tile is not None and tile.get("kind") == "PLANT" and tile.get("watered_today") == False: # Comprobación de si se ha regado
        farmer = ["WATER"] # Se riega la semilla
    elif tile is not None and tile.get("kind") == "PLANT" and obs["day"] - tile["planted_day"] >= 2: # Comprueba si han pasado más de 2 días desde que se plantó para cosechar.
        farmer = ["HARVEST"] # Se cosecha la semilla
    else: 
        farmer = ["PASS"]
    return {"farmer": farmer, "hands": [], "market": market}

In [5]:
def score_episodes(episode_results):
    """Aggregate (my_money, opponent_money) tuples into win_rate and avg_money.

    A tie counts as half a win (0.5) toward win_rate.
    """
    scores = []
    my_moneys = []
    for my_money, opponent_money in episode_results:
        my_moneys.append(my_money)
        if my_money > opponent_money:
            scores.append(1)
        elif my_money == opponent_money:
            scores.append(0.5)
        else:
            scores.append(0)

    return {
        "win_rate": sum(scores) / len(scores),
        "avg_money": sum(my_moneys) / len(my_moneys),
    }

In [16]:
from kaggle_environments import make

env = make("kaggriculture", configuration={"episodeSteps": 720}, debug=True)

episode_results = []
for i in range(10):
    result = env.run([my_agent, "starter"])
    episode_result = result[-1][0]['reward'], result[-1][1]['reward']
    episode_results.append(episode_result)
score_episodes(episode_results)

{'win_rate': 1.0, 'avg_money': 3920.0}

In [18]:
def run_episodes(agent0, agent1, n=10, env=None):
    """Run n episodes of agent0 vs agent1, collecting each episode's final (agent0_money, agent1_money)."""
    results = []
    if env is None:
        env = make("kaggriculture", configuration={"episodeSteps": 720}, debug=True)
    for _ in range(n):
        steps = env.run([agent0, agent1])
        results.append((steps[-1][0]["reward"], steps[-1][1]["reward"]))
    return results

score_episodes(run_episodes(my_agent, "starter"))   

{'win_rate': 1.0, 'avg_money': 3843.0}

In [20]:
def evaluate(agent0, agent1, n=10, env=None):
    score = score_episodes(run_episodes(agent0, agent1, n, env))
    return score
evaluate(my_agent, "starter")

{'win_rate': 1.0, 'avg_money': 3903.0}